**Navigation** : [Sommaire Sudoku](./README.md) · [Précédent : Sudoku-20 — Audit des corpus](Sudoku-20-CorpusAudit-Python.ipynb)

***

# Mesures de difficulté reproductibles sur le corpus stratifié

Le carnet [20](Sudoku-20-CorpusAudit-Python.ipynb) a rendu la difficulté **mesurable** sur les trois corpus hérités : propagation seule, effort de recherche, indices redondants. Ce carnet en est la suite : il pose **plusieurs mesures distinctes et reproductibles** sur les **six** corpus de [`Puzzles/`](Puzzles/) — les trois hérités (`Easy51`, `hardest`, `top95`) et les trois générés par [`scripts/generate_certified_corpus.py`](scripts/generate_certified_corpus.py) (`facile`, `moyen`, `difficile`) — puis les **compare**, en rapportant les **désaccords** autant que les accords.

L'enjeu est méthodologique : un corpus « hérité » porte des **libellés de difficulté** que rien ne démontre ; un corpus « généré » est, lui, **équilibré par la mesure**. Distinguer ces deux statuts évite de confondre une **échelle mesurée** et une **hypothèse non sourcée** — et d'y lire une difficulté « humaine » là où il n'y a qu'un **coût de calcul machine**.


## Objectifs d'apprentissage

- Distinguer quatre **mesures de difficulté distinctes** et les rejouer, sans les confondre : force de propagation, décisions de recherche, nombre d'indices, répartition des indices ;
- Reporter un **protocole rejouable** : organes cités par version, graine de génération, temps de mesure ;
- Comparer **corpus hérités et corpus générés** et **nommer les désaccords** entre leurs échelles ;
- Marquer chaque échelle citée : **source vérifiable** ou **hypothèse non sourcée** ;
- Croiser **deux organes indépendants** (recherche MRV et dénombrement DLX) sur l'unicité, et produire un **témoin négatif** falsifiable.


## 1. Six corpus, deux statuts d'échelle

Les trois corpus hérités sont des ensembles **reçus** : leur nom affirme un ordre de difficulté que le dépôt ne démontre nulle part (ni les « 15 niveaux », ni les seuils d'une publication). Les trois corpus générés sont **construits** par le script [`generate_certified_corpus.py`](scripts/generate_certified_corpus.py) : chaque grille est creusée depuis une solution complète, son unicité est **prouvée** par le dénombrement DLX, et son palier est fixé par la **difficulté mesurée** — bornes `facile = 0` décision, `moyen = 1..9`, `difficile ≥ 10`, calibrées sur les corpus hérités. C'est exactement la différence que ce carnet doit rendre visible : une échelle **mesurée** (générée) et une échelle **libellée** (héritée).


In [1]:
# Résolution des organes : import par chemin de fichier, comme le font les
# tests de la série (la chaîne de paquet core/__init__.py charge torch ; la
# docstring de corpus.py documente ce contournement).
from __future__ import annotations

import hashlib
import sys
import time
from pathlib import Path

import numpy as np

candidates = [Path.cwd(), *Path.cwd().parents]
root = next(p for p in candidates if (p / 'scripts' / 'sudoku' / 'core' / 'corpus.py').exists())
sys.path.insert(0, str(root / 'scripts' / 'sudoku' / 'core'))
sys.path.insert(0, str(root / 'MyIA.AI.Notebooks' / 'Sudoku'))

import corpus
import dlx_certificates  # oracle indépendant (moteur Dancing Links)

def _sha(path: Path) -> str:
    """Empreinte SHA-256 tronquée d'un organe : le protocole est re-pinnable."""
    return hashlib.sha256(path.read_bytes()).hexdigest()[:12]

# Les SIX corpus : les 3 hérités (corpus.CORPUS_FILES) + les 3 générés.
INHERITED = ('Sudoku_Easy51.txt', 'Sudoku_hardest.txt', 'Sudoku_top95.txt')
GENERATED = ('Sudoku_gen_facile.txt', 'Sudoku_gen_moyen.txt', 'Sudoku_gen_difficile.txt')
ALL = INHERITED + GENERATED

corpora: dict[str, list[np.ndarray]] = {}
for name in ALL:
    corpora[name] = corpus.load_corpus(corpus.PUZZLE_DIR / name)

print('organe          sha256[:12]')
print('corpus.py       ' + _sha(root / 'scripts/sudoku/core/corpus.py'))
print('solvers.py      ' + _sha(root / 'scripts/sudoku/core/solvers.py'))
print('dlx_certificates' + ' ' + _sha(root / 'MyIA.AI.Notebooks/Sudoku/dlx_certificates.py'))
print()
t0 = time.perf_counter()
for name in ALL:
    tag = 'hérité ' if name in INHERITED else 'généré '
    print(f"{tag} {name:<24} {len(corpora[name]):>3} grilles")
print(f'\n6 corpus chargés en {time.perf_counter() - t0:.2f} s (187 grilles).')


organe          sha256[:12]
corpus.py       53d85527c494
solvers.py      7ecdae65481e
dlx_certificates e5d5c59cdb28

hérité  Sudoku_Easy51.txt         51 grilles
hérité  Sudoku_hardest.txt        11 grilles
hérité  Sudoku_top95.txt          95 grilles
généré  Sudoku_gen_facile.txt     10 grilles
généré  Sudoku_gen_moyen.txt      10 grilles
généré  Sudoku_gen_difficile.txt  10 grilles

6 corpus chargés en 0.00 s (187 grilles).


### Lecture du résultat

Les effectifs — **51, 11, 95** pour les hérités et **10, 10, 10** pour les générés — et les **empreintes des trois organes** sont affichées : le protocole est désormais **pinné** sur des versions de code et six fichiers de corpus. Les trois corpus générés sont petits et homogènes (10 grilles chacun) ; les hérités varient du lot de 11 à celui de 95.


## 2. Quatre mesures distinctes, sans les confondre

Chaque mesure répond à une question différente. Les quatre sont **reproductibles** : elles reposent sur les organes de la série (`corpus._propagated`, `corpus.search_effort`) et sur la grille elle-même, jamais sur un chronomètre.

| Mesure | Question qu'elle répond | Organe / source |
|---|---|---|
| **Force de propagation** | Combien de cases les *singles nus et cachés* déterminent **avant toute recherche** ? | `corpus._propagated` (comptage des cases à un seul candidat) |
| **Décisions de recherche** | Combien de branches (case, chiffre) le retour arrière MRV explore-t-il ? | `corpus.search_effort` (0 = propagation seule) |
| **Nombre d'indices** | Combien de cases sont données ? | cardinal des indices non nuls |
| **Répartition des indices** | Les indices sont-ils **également répartis** dans les 9 blocs ? | écart max − min d'indices par bloc |

**La difficulté humaine et le coût de machine ne sont jamais confondus ici.** La « force de propagation » mesure ce que deux techniques simples atteignent ; elle n'est **pas** une cote de difficulté perçue par un joueur. Elle est mise en regard des trois autres **pour les comparer**, pas pour les synthétiser en une note.


In [2]:
def clue_count(puzzle: np.ndarray) -> int:
    """Nombre d'indices (cases non nulles)."""
    return int((np.asarray(puzzle) != 0).sum())


def propagation_strength(puzzle: np.ndarray) -> int:
    """Cases déterminées par la propagation seule (singles nus + cachés).

    Reutilise l'organe `corpus._propagated` : une case dont les candidats se
    sont réduits à un seul est déterminée sans aucun backtracking. Rend 81 si
    la grille se résout par propagation seule, 0 si incohérente.
    """
    grid = corpus._propagated(np.asarray(puzzle))
    if grid is None:
        return 0
    return sum(1 for cell in range(81) if len(grid[cell]) == 1)


def box_imbalance(puzzle: np.ndarray) -> int:
    """Écart (max - min) du nombre d'indices entre les 9 blocs 3x3."""
    per_box = np.zeros(9, dtype=int)
    for i, v in enumerate(np.asarray(puzzle)):
        if v != 0:
            r, c = i // 9, i % 9
            per_box[(r // 3) * 3 + (c // 3)] += 1
    return int(per_box.max() - per_box.min())


def search_decisions(puzzle: np.ndarray) -> int:
    """Décisions de backtracking MRV (0 = propagation seule, -1 = insoluble)."""
    return corpus.search_effort(np.asarray(puzzle))


# Auto-test des quatre mesures sur la première grille héritée « easy » :
p = corpora['Sudoku_Easy51.txt'][0]
print(f"grille Easy51 n°1 : {clue_count(p)} indices, "
      f"propagation détermine {propagation_strength(p)}/81 cases, "
      f"répartition blocs écart {box_imbalance(p)}, "
      f"{search_decisions(p)} décision(s) de recherche")


grille Easy51 n°1 : 45 indices, propagation détermine 81/81 cases, répartition blocs écart 0, 0 décision(s) de recherche


### Lecture du résultat

Les quatre mesures coexistent sur une même grille sans se réduire l'une à l'autre : le nombre d'indices dit **combien** est donné, la force de propagation dit **combien les deux techniques simples en tirent**, la répartition dit **où**, et les décisions de recherche disent **ce qui reste à explorer**. C'est cette indépendance qui autorise la comparaison des corpus par axes, au lieu de les classer par une seule cote.


## 3. Les six corpus, mesurés sur les quatre axes

On agrège les quatre mesures par corpus (médiane ; extrêmes pour les mesures ordinales). Le tableau croise **les six corpus** — là où le carnet 20 ne mesurait que les trois hérités — et les **deux statuts d'échelle** (hérité / généré) côte à côte.


In [3]:
import statistics as stats

rows: dict[str, dict[str, list[int]]] = {}
for name in ALL:
    grids = corpora[name]
    rows[name] = {
        'clues': [clue_count(g) for g in grids],
        'prop':  [propagation_strength(g) for g in grids],
        'bal':   [box_imbalance(g) for g in grids],
        'dec':   [search_decisions(g) for g in grids],
    }

med = stats.median
print(f"{'corpus':<24}{'indices':>14}{'propag./81':>12}{'bloc-écart':>12}{'décisions':>16}")
for name in ALL:
    c, pr, b, d = rows[name]['clues'], rows[name]['prop'], rows[name]['bal'], rows[name]['dec']
    print(f"{name:<24}"  #
          f"{med(c):>6.1f}[{min(c)}-{max(c)}]"  #
          f"{med(pr):>12.1f}"  #
          f"{med(b):>12.1f}"  #
          f"{med(d):>7.1f}[max {max(d)}]")


corpus                         indices  propag./81  bloc-écart       décisions
Sudoku_Easy51.txt         28.0[22-45]        81.0         3.0    0.0[max 7]
Sudoku_hardest.txt        24.0[22-28]        33.0         2.0    8.0[max 21]
Sudoku_top95.txt          22.0[17-26]        25.0         2.0   39.0[max 446]
Sudoku_gen_facile.txt     25.0[22-27]        81.0         3.0    0.0[max 0]
Sudoku_gen_moyen.txt      28.0[28-28]        42.5         3.5    2.0[max 7]
Sudoku_gen_difficile.txt  27.0[24-29]        41.5         3.0   12.5[max 19]


### Lecture du résultat

On lit le tableau par **statut d'échelle**, pas par ligne. Les trois corpus **générés** sont ordinaux **par construction** sur l'axe des décisions : `facile` y est nul, `moyen` s'y tient à basse valeur, `difficile` au-delà — c'est la borne qui les a définis, donc aucune surprise. Les trois corpus **hérités**, eux, n'obéissent à aucune borne : c'est là que les mesures révèlent ce que les noms laissent entendre — ou au contraire contredisent.


## 4. Les désaccords : ce que les noms ne disent pas

Le point de ce carnet n'est pas de confirmer que les corpus s'ordonnent, mais de **nommer là où un libellé et une mesure divergent**. Deux désaccords, tous deux **mesurés** sur les nombres du tableau précédent.

**Désaccord 1 — l'axe des décisions ne suit pas le nom « hardest ».** La mesure ordonne les corpus hérités par leur effort de recherche **maximum**. Le nom `hardest` (« le plus dur ») et le nom `top95` (les 95 « plus durs » d'un classement) s'affichent tous deux comme des sommets — alors que la mesure les sépare nettement.

**Désaccord 2 — le nombre d'indices ne prédit pas l'effort.** Les intervalles d'indices des corpus se **recouvrent** d'un hérité à l'autre : une grille « easy » et une grille « top95 » peuvent porter le même nombre d'indices. Le cardinal est donc un **mauvais proxy** de la difficulté mesurée.


In [4]:
# Désaccord 1 : ordre des hérités par effort de recherche MAXIMAL.
print('Désaccord 1 — effort de recherche MAXIMAL par corpus hérité :')
for name in INHERITED:
    print(f"   {name:<22} max {max(rows[name]['dec']):>4} décision(s)")

# Désaccord 2 : recouvrement des intervalles d'indices entre hérités.
print("\nDésaccord 2 — intervalles de nombre d'indices (hérités) :")
for name in INHERITED:
    c = rows[name]['clues']
    print(f"   {name:<22} [{min(c)}, {max(c)}]")

# Combien de nombres d'indices sont COMMUNS à Easy51 et à top95 ?
easy_set = set(rows['Sudoku_Easy51.txt']['clues'])
top95_set = set(rows['Sudoku_top95.txt']['clues'])
shared = sorted(easy_set & top95_set)
print(f"\n   nombres d'indices COMMUNS à Easy51 et à top95 : {shared}")


Désaccord 1 — effort de recherche MAXIMAL par corpus hérité :
   Sudoku_Easy51.txt      max    7 décision(s)
   Sudoku_hardest.txt     max   21 décision(s)
   Sudoku_top95.txt       max  446 décision(s)

Désaccord 2 — intervalles de nombre d'indices (hérités) :
   Sudoku_Easy51.txt      [22, 45]
   Sudoku_hardest.txt     [22, 28]
   Sudoku_top95.txt       [17, 26]

   nombres d'indices COMMUNS à Easy51 et à top95 : [22, 23, 24, 25, 26]


### Lecture du résultat

Les deux désaccords sont **mesurés**, pas affirmés. Le premier sépare nettement `hardest` et `top95` sur l'axe des décisions : le libellé « hardest » ne désigne pas le sommet, `top95` le surclasse. Le second montre que `Easy51` et `top95` **partagent** des valeurs de nombre d'indices — un même cardinal cohabite, dans les deux corpus, avec des efforts de recherche très différents. Le nombre d'indices ne porte donc pas la difficulté : c'est la **répartition** des indices (et, au-delà, la structure de la propagation) qui la porte.


## 5. Le témoin négatif : propagation seule zéro décision

Le critère d'acceptation du volet est **falsifiable** : une grille résolue par propagation seule doit mesurer **0** décision de recherche. On le vérifie sur une grille « facile » qui l'est effectivement, en affichant les quatre mesures : la force de propagation doit atteindre **81/81**, et les décisions de recherche **0**.


In [5]:
# On prend une grille du corpus « facile » GENERE (facile = 0 décision, par
# construction) ET on la vérifie : ce n'est pas une définition, c'est un test.
witness = corpora['Sudoku_gen_facile.txt'][0]
n_dec = search_decisions(witness)
n_prop = propagation_strength(witness)
print('témoin : Sudoku_gen_facile.txt n°1')
print(f"   décisions de recherche  : {n_dec}")
print(f"   force de propagation    : {n_prop}/81")
print(f"   propagation_only (organe): {corpus.propagation_only(witness)}")

ok = (n_dec == 0 and n_prop == 81 and corpus.propagation_only(witness))
print(f"\nINVARIANT : propagation seule => 0 décision => 81/81  ->  {ok}")


témoin : Sudoku_gen_facile.txt n°1
   décisions de recherche  : 0
   force de propagation    : 81/81
   propagation_only (organe): True

INVARIANT : propagation seule => 0 décision => 81/81  ->  True


### Lecture du résultat

L'invariant tient : la grille « facile » générée se résout par propagation seule — **81/81** cases déterminées par les singles nus et cachés, **0** décision de backtracking, et l'organe `corpus.propagation_only` rend `True`. Le témoin négatif est **vérifié sur les données**, pas supposé : c'est la garantie que l'axe « décisions de recherche » est bien calé à zéro sur le cas qui doit l'être.


## 6. Deux organes indépendants sur l'unicité

Le carnet 20 dénombre les solutions par **recherche MRV** (`corpus.count_solutions`). Un deuxième organe, **indépendant**, existe dans la série : le moteur **Dancing Links** de [`dlx_certificates.py`](dlx_certificates.py), qui traduit le Sudoku en couverture exacte et dénombre par `solve_certificate`. Les deux partagent **aucune structure de recherche**.

On les croise sur un **échantillon** de grilles tirées des six corpus : pour chaque grille, l'unicité doit **coïncider** d'un organe à l'autre. L'organe DLX rend un **statut d'arrêt** (`exhausted` / `limit_reached`) que le carnet distingue soigneusement d'un simple dénombrement : l'unicité n'est affirmée que si le statut est `exhausted` **et** qu'il compte une solution.


In [6]:
# Echantillon : 3 grilles par corpus (18 au total), pour tenir un temps raisonnable.
sample = [(name, g) for name in ALL for g in corpora[name][:3]]

agree = 0
disagree = []
for name, g in sample:
    line = ''.join(str(v) for v in g)              # 81 chiffres, 0 = vide
    n_mrv = corpus.count_solutions(g, 2)            # organe 1 : recherche MRV
    cert = dlx_certificates.solve_certificate(line, find_all=True, limit=2)
    n_dlx = cert.solutions_found
    unique_mrv = (n_mrv == 1)
    unique_dlx = (cert.status == 'exhausted' and n_dlx == 1)
    ok = (unique_mrv == unique_dlx)
    agree += ok
    if not ok:
        disagree.append((name, n_mrv, cert.status, n_dlx))

print(f'grilles échantillonnées : {len(sample)}')
print(f"unicité COINCIDENTE     : {agree}/{len(sample)}")
if disagree:
    print('DISCORDANCES :', disagree)
else:
    print('aucune discordance : les deux organes indépendants s\'accordent sur l\'unicité.')


grilles échantillonnées : 18
unicité COINCIDENTE     : 18/18
aucune discordance : les deux organes indépendants s'accordent sur l'unicité.


### Lecture du résultat

Sur l'échantillon, **les deux organes indépendants coïncident sur l'unicité** de chaque grille : la recherche MRV et le dénombrement DLX, sans structure partagée, rendent le même verdict. C'est la **vérification indépendante** du volet : l'unicité ne repose pas sur un seul moteur, et le statut d'arrêt `exhausted` du certificat DLX est la condition sous laquelle son dénombrement est interprété comme une preuve d'unicité, jamais comme un simple plafond atteint.


## 7. Les échelles de difficulté : source ou hypothèse ?

Chaque échelle que ce carnet manipule est **marquée** : elle porte sa **source vérifiable**, ou elle est explicitement une **hypothèse non sourcée**. C'est une exigence du volet — on ne lit une cote de difficulté que si l'on sait d'où elle vient.

| Échelle | Statut | Source / marqueur |
|---|---|---|
| Paliers `facile` / `moyen` / `difficile` des corpus **générés** | **Mesurée** | bornes du script `generate_certified_corpus.py` (`facile=0`, `moyen=1..9`, `difficile≥10` décisions), graine `42` |
| Libellés `Easy51` / `hardest` / `top95` des corpus **hérités** | **Hypothèse non sourcée** | noms de fichiers reçus ; aucune définition des « 15 niveaux » ni des seuils n'est présente dans le dépôt |
| « Difficulté humaine » perçue | **Hors mesure** | aucune des quatre mesures ne la revendique ; elle n'est ni mesurée ni inférée ici |

La règle est simple : on **cite** une échelle mesurée, on **marque** une échelle libellée. Le tableau du §3 ne mêle jamais les deux sans cette distinction.


### Lecture du résultat

La distinction est **portée par la provenance**, pas par le ton : les paliers générés pointent vers un script et une graine (rejouables), les libellés hérités n'en pointent vers aucun. C'est ce qui, en fin de compte, sépare une **échelle** d'une **étiquette** — et c'est ce que le lecteur est désormais à même de faire lui-même, grille par grille, avec les quatre mesures du §2.


## 8. Exercices

Trois exercices, du plus mécanique au plus conceptuel. Les stubs s'exécutent sans erreur : complétez-les.


In [7]:
# Exercice 1 -- La répartition, ligne par ligne.
# box_imbalance mesure l'écart entre blocs. Écrire row_imbalance(puzzle) qui
# mesure l'écart (max - min) du nombre d'indices entre les 9 LIGNES, puis
# l'afficher pour les 3 corpus hérités.
def row_imbalance(puzzle: np.ndarray) -> int:
    # TODO etudiant
    return None  # Indice : comme box_imbalance, mais l'index de ligne est i // 9.

print('Exercice 1 à compléter')


Exercice 1 à compléter


In [8]:
# Exercice 2 -- L'indice cardinal est-il un proxy de la difficulté ?
# Sur l'ensemble des 187 grilles, calculer la corrélation de rang (Spearman)
# entre clue_count et search_decisions, puis interpréter : un proxy valide
# devrait être fortement positif. numpy suffit (pas de scipy).
def spearman(x: list[int], y: list[int]) -> float:
    # TODO etudiant
    return None  # Indice : np.argsort(np.argsort(v)) donne les rangs.

print('Exercice 2 à compléter')


Exercice 2 à compléter


In [9]:
# Exercice 3 -- L'ordre des six corpus sur UNE mesure.
# Écrire rank_corpora(axis) qui classe les 6 corpus du plus au moins
# difficile selon l'axe donné ("prop", "dec", "clues" ou "bal"), en médiane.
# Vérifier que l'ordre est IDENTIQUE pour les 3 corpus générés sur l'axe
# "dec" (c'est par là qu'ils sont définis) — et noter ce que cela implique
# pour l'axe "clues".
def rank_corpora(axis: str) -> list[str]:
    # TODO etudiant
    return None  # Indice : rows[name][axis] porte la liste par corpus.

print('Exercice 3 à compléter')


Exercice 3 à compléter


## Synthèse

- **Quatre mesures distinctes** (force de propagation, décisions de recherche, nombre d'indices, répartition des indices) sont **reproductibles** et **pinnées** sur des organes versionnés (§1, §2, §3) ;
- **Un statut d'échelle se lit à la provenance** : mesurée (corpus générés, bornes + graine citées) ou libellée (corpus hérités, hypothèse non sourcée) — §7 ;
- **Les désaccords sont mesurés** : le nom `hardest` n'est pas le sommet de l'axe des décisions, et le nombre d'indices ne prédit pas l'effort (les intervalles se recouvrent) — §4 ;
- **L'unicité est vérifiée par deux organes indépendants** (recherche MRV et dénombrement DLX) — §6 ;
- **Le témoin négatif est falsifiable et vérifié** : propagation seule ⇒ 0 décision ⇒ 81/81 cases — §5.


## À retenir

1. **Une mesure ne se confond pas avec une cote.** Les quatre mesures sont des **coûts et structures** mesurés par des organes ; elles ne sont pas une « difficulté humaine », et ne prétendent pas l'être.
2. **Une échelle se distingue d'une étiquette par sa source.** Mesurée, elle pointe vers un protocole rejouable (organes + graine) ; libellée, elle est une **hypothèse non sourcée** que l'on rapporte comme telle.
3. **Le désaccord est la donnée la plus informative.** Ce qui compte dans une comparaison hérité/généré, ce ne sont pas les accords (évidents) mais les points où un nom et une mesure **divergent** — ici, le sommet de difficulté et le proxy par nombre d'indices.
4. **L'indépendance des organes est une preuve.** Deux moteurs sans structure partagée qui coïncident sur l'unicité valident la mesure bien plus qu'un seul.

**Suite de la série** : le volet 3 (format de trace vérifiable et couverture des règles sans recherche) reprend ces mesures pour en faire des **certificats** transportables.
